# 08 - Modeling v2: Uji Cepat CatBoost dengan Fitur Baru

Menguji apakah 508 fitur baru dari Feature Engineering v2 (agregasi 6 tabel
tambahan) meningkatkan performa CatBoost dibanding v1 (CV ROC-AUC 0.7594).
Hanya CatBoost dulu (model terbaik v1) sebagai validasi cepat sebelum
retrain 6 model lain.

#  Import


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
from modeling.train import (
    load_featured_dataset, load_feature_metadata,
    create_or_load_holdout_split, create_or_load_cv_folds,
    train_catboost,train_lightgbm,train_xgboost,tune_catboost_hyperparameters,
)
from features_v2 import build_feature_metadata_v2, save_feature_metadata_v2

## Load Dataset v2, Metadata v1, dan Split/Folds yang Sudah Ada

`holdout_split.json` dan `cv_folds.json` TIDAK dibuat ulang -- SK_ID_CURR dan
jumlah baris v2 identik dengan v1, jadi split lama tetap valid dan adil untuk
perbandingan v1 vs v2.

In [2]:
df_v2 = load_featured_dataset(path=config.PROCESSED_DATA_DIR / "application_train_featured_v2.csv")
metadata_v1 = load_feature_metadata()

split = create_or_load_holdout_split(df_v2)   # akan load yang sudah ada, bukan buat baru
folds = create_or_load_cv_folds(df_v2, split)  # sama, load yang sudah ada

print("Shape dataset v2:", df_v2.shape)

Shape dataset v2: (307511, 581)


## Bangun Metadata Fitur v2 (Gabung 73 Fitur v1 + 508 Fitur Baru)

In [3]:
metadata_v2 = build_feature_metadata_v2(df_v2, metadata_v1)
save_feature_metadata_v2(metadata_v2)

print("Jumlah tree_features_v2       :", len(metadata_v2["tree_features_v2"]))
print("Jumlah linear_mlp_features_v2 :", len(metadata_v2["linear_mlp_features_v2"]))

Jumlah tree_features_v2       : 574
Jumlah linear_mlp_features_v2 : 574


## Retrain CatBoost dengan Fitur v2

`model_name="catboost_v2"` supaya hasil tersimpan terpisah, tidak menimpa
hasil CatBoost v1 di `models/catboost/`.

In [4]:
metadata_for_catboost = {"tree_features": metadata_v2["tree_features_v2"]}

result_cb_v2 = train_catboost(df_v2, metadata_for_catboost, split, folds, model_name="catboost_v2")

print("=== CatBoost v2 ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_cb_v2["cv_summary"]["roc_auc_mean"], result_cb_v2["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_cb_v2["cv_summary"]["pr_auc_mean"], result_cb_v2["cv_summary"]["pr_auc_std"]
))
print("\n--- Perbandingan dengan v1 ---")
print("CatBoost v1: ROC-AUC 0.7594 ± 0.0016 | PR-AUC 0.2436 ± 0.0059")

d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()


=== CatBoost v2 ===
CV ROC-AUC: 0.7795 ± 0.0021
CV PR-AUC : 0.2690 ± 0.0068

--- Perbandingan dengan v1 ---
CatBoost v1: ROC-AUC 0.7594 ± 0.0016 | PR-AUC 0.2436 ± 0.0059


## Retrain LightGBM & XGBoost dengan Fitur v2

Menguji apakah kenaikan skor dari fitur baru juga berlaku untuk model boosting
lain, sekaligus menyiapkan kandidat untuk tahap ensemble/stacking nanti.

#  LightGBM v2


In [5]:
metadata_for_boosting = {"tree_features": metadata_v2["tree_features_v2"]}

result_lgb_v2 = train_lightgbm(df_v2, metadata_for_boosting, split, folds, model_name="lightgbm_v2")

print("=== LightGBM v2 ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_lgb_v2["cv_summary"]["roc_auc_mean"], result_lgb_v2["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_lgb_v2["cv_summary"]["pr_auc_mean"], result_lgb_v2["cv_summary"]["pr_auc_std"]
))
print("\n--- Perbandingan dengan v1 ---")
print("LightGBM v1: ROC-AUC 0.7571 ± 0.0010 | PR-AUC 0.2411 ± 0.0052")

d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Home

=== LightGBM v2 ===
CV ROC-AUC: 0.7775 ± 0.0019
CV PR-AUC : 0.2664 ± 0.0061

--- Perbandingan dengan v1 ---
LightGBM v1: ROC-AUC 0.7571 ± 0.0010 | PR-AUC 0.2411 ± 0.0052


# XGBoost v2

In [6]:
result_xgb_v2 = train_xgboost(df_v2, metadata_for_boosting, split, folds, model_name="xgboost_v2")

print("=== XGBoost v2 ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_xgb_v2["cv_summary"]["roc_auc_mean"], result_xgb_v2["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_xgb_v2["cv_summary"]["pr_auc_mean"], result_xgb_v2["cv_summary"]["pr_auc_std"]
))
print("\n--- Perbandingan dengan v1 ---")
print("XGBoost v1: ROC-AUC 0.7545 ± 0.0010 | PR-AUC 0.2388 ± 0.0029")

d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()


=== XGBoost v2 ===
CV ROC-AUC: 0.7733 ± 0.0027
CV PR-AUC : 0.2606 ± 0.0069

--- Perbandingan dengan v1 ---
XGBoost v1: ROC-AUC 0.7545 ± 0.0010 | PR-AUC 0.2388 ± 0.0029


## Cek Ensemble Sederhana (Rata-rata OOF, Tanpa Retrain)

Menghitung ROC-AUC/PR-AUC dari rata-rata prediksi out-of-fold (OOF) CatBoost v2,
LightGBM v2, dan XGBoost v2 -- tanpa perlu melatih model baru, karena OOF sudah
tersimpan dari hasil CV yang sudah dijalankan.

In [7]:
import numpy as np
from modeling.train import evaluate_predictions

oof_true = result_cb_v2["oof_true"]

assert np.allclose(oof_true, result_lgb_v2["oof_true"], equal_nan=True), "oof_true CatBoost & LightGBM tidak sama!"
assert np.allclose(oof_true, result_xgb_v2["oof_true"], equal_nan=True), "oof_true CatBoost & XGBoost tidak sama!"

oof_proba_cb = result_cb_v2["oof_proba"]
oof_proba_lgb = result_lgb_v2["oof_proba"]
oof_proba_xgb = result_xgb_v2["oof_proba"]

oof_proba_ensemble_equal = (oof_proba_cb + oof_proba_lgb + oof_proba_xgb) / 3

metrics_ensemble = evaluate_predictions(oof_true, oof_proba_ensemble_equal, threshold=0.5)

print("=== Ensemble Rata-rata Sederhana (CatBoost + LightGBM + XGBoost) ===")
print("ROC-AUC:", round(metrics_ensemble["roc_auc"], 4))
print("PR-AUC :", round(metrics_ensemble["pr_auc"], 4))

print("\n--- Perbandingan ---")
print("CatBoost v2 (sendiri): ROC-AUC 0.7795")
print("LightGBM v2 (sendiri): ROC-AUC 0.7775")
print("XGBoost v2  (sendiri): ROC-AUC 0.7733")

=== Ensemble Rata-rata Sederhana (CatBoost + LightGBM + XGBoost) ===
ROC-AUC: 0.7797
PR-AUC : 0.2689

--- Perbandingan ---
CatBoost v2 (sendiri): ROC-AUC 0.7795
LightGBM v2 (sendiri): ROC-AUC 0.7775
XGBoost v2  (sendiri): ROC-AUC 0.7733


# Ensemble Berbobot (Opsional, Beri Bobot Lebih ke Model Terbaik)

In [8]:
weights = {"cb": 0.5, "lgb": 0.3, "xgb": 0.2}  # total harus 1.0

oof_proba_ensemble_weighted = (
    weights["cb"] * oof_proba_cb +
    weights["lgb"] * oof_proba_lgb +
    weights["xgb"] * oof_proba_xgb
)

metrics_ensemble_weighted = evaluate_predictions(oof_true, oof_proba_ensemble_weighted, threshold=0.5)

print("=== Ensemble Berbobot (CatBoost 50% / LightGBM 30% / XGBoost 20%) ===")
print("ROC-AUC:", round(metrics_ensemble_weighted["roc_auc"], 4))
print("PR-AUC :", round(metrics_ensemble_weighted["pr_auc"], 4))

=== Ensemble Berbobot (CatBoost 50% / LightGBM 30% / XGBoost 20%) ===
ROC-AUC: 0.7804
PR-AUC : 0.27


## Hyperparameter Tuning CatBoost v2 (Optuna)

Mencari kombinasi learning_rate, depth, l2_leaf_reg, bagging_temperature, dan
random_strength yang optimal. 25 trial (bisa disesuaikan) memakai split cepat
80/20, lalu divalidasi ulang dengan CV penuh.

In [9]:
tuning_result = tune_catboost_hyperparameters(df_v2, metadata_for_catboost, split, n_trials=25)

print("Best ROC-AUC (split cepat):", round(tuning_result["best_value"], 4))
print("Best params:", tuning_result["best_params"])

[I 2026-09-01 18:06:45,949] A new study created in memory with name: no-name-28323db3-f0e2-4f5c-87ff-236c5d565e01


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-09-01 18:25:06,802] Trial 0 finished with value: 0.7701712952417976 and parameters: {'learning_rate': 0.023688639503640783, 'depth': 10, 'l2_leaf_reg': 5.395030966670228, 'bagging_temperature': 0.5986584841970366, 'random_strength': 0.7800932022121826}. Best is trial 0 with value: 0.7701712952417976.
[I 2026-09-01 18:27:42,293] Trial 1 finished with value: 0.7596144379215195 and parameters: {'learning_rate': 0.01432169828911152, 'depth': 4, 'l2_leaf_reg': 7.348118405270449, 'bagging_temperature': 0.6011150117432088, 'random_strength': 3.540362888980227}. Best is trial 0 with value: 0.7701712952417976.
[I 2026-09-01 18:47:50,779] Trial 2 finished with value: 0.7670519303931411 and parameters: {'learning_rate': 0.010485387725194618, 'depth': 10, 'l2_leaf_reg': 6.798962421591129, 'bagging_temperature': 0.21233911067827616, 'random_strength': 0.9091248360355031}. Best is trial 0 with value: 0.7701712952417976.
[I 2026-09-01 18:51:32,896] Trial 3 finished with value: 0.7689045683258

## Validasi Ulang: Full CV 5-Fold dengan Hyperparameter Terbaik

`model_name="catboost_v2_tuned"` supaya hasil ini tersimpan terpisah dari
CatBoost v2 versi default sebelumnya.

In [10]:
result_cb_v2_tuned = train_catboost(
    df_v2, metadata_for_catboost, split, folds,
    model_name="catboost_v2_tuned",
    extra_params=tuning_result["best_params"],
)

print("=== CatBoost v2 Tuned ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_cb_v2_tuned["cv_summary"]["roc_auc_mean"], result_cb_v2_tuned["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_cb_v2_tuned["cv_summary"]["pr_auc_mean"], result_cb_v2_tuned["cv_summary"]["pr_auc_std"]
))
print("\n--- Perbandingan ---")
print("CatBoost v2 (default) : ROC-AUC 0.7795")
print("CatBoost v1           : ROC-AUC 0.7594")

d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()


=== CatBoost v2 Tuned ===
CV ROC-AUC: 0.7795 ± 0.0019
CV PR-AUC : 0.2684 ± 0.0067

--- Perbandingan ---
CatBoost v2 (default) : ROC-AUC 0.7795
CatBoost v1           : ROC-AUC 0.7594


In [11]:
from modeling.train import display_model_comparison
display_model_comparison()

,Model,Jalur Fitur,N Fitur,ROC-AUC (mean),ROC-AUC (std),PR-AUC (mean),PR-AUC (std),Best Iter,Waktu Training (s)
rank,,,,,,,,,
1,catboost_v2_tuned,tree,574,0.7795,0.0019,0.2684,0.0067,719.4,841.1
2,catboost_v2,tree,574,0.7795,0.0021,0.2690,0.0068,738.6,891.7
3,catboost_v2,tree,574,0.7795,0.0021,0.2690,0.0068,738.6,962.6
4,catboost_v2,tree,574,0.7795,0.0021,0.2690,0.0068,738.6,856.9
5,catboost_v2,tree,574,0.7795,0.0021,0.2690,0.0068,738.6,881.7
6,lightgbm_v2,tree,693,0.7775,0.0019,0.2664,0.0061,236.8,114.1
7,lightgbm_v2,tree,693,0.7775,0.0019,0.2664,0.0061,236.8,125.8
8,xgboost_v2,tree,693,0.7733,0.0027,0.2606,0.0069,278.2,396.0
9,xgboost_v2,tree,693,0.7733,0.0027,0.2606,0.0069,278.2,369.6
